In [ ]:
# Importing the necessary libraries
import requests
import pandas as pd
import time

# List of subreddits to fetch data from
subreddits = [
    "programming", "learnprogramming", "Python", "java", "javascript",
    "learnpython", "webdev", "coding", "technology", "androiddev",
    "GenAI4all", "generativeAI", "OpenAI", "linux", "C_Programming",
    "CyberCrime", "cybersecurity", "softwareupdates", "datascience",
    "dataanalysis", "dataengineering", "algorithms", "analytics",
    "MachineLearning", "learnmachinelearning", "deeplearning",
    "MachineLearningAndAI", "automation", "aiengineering",
    "neuralnetworks"
]

# API endpoints for fetching submissions and comments
SUBMISSION_URL = "https://api.pullpush.io/reddit/search/submission/"
COMMENT_URL = "https://api.pullpush.io/reddit/search/comment/"

# Parameters for data collection
posts_per_subreddit = 50
comments_per_post = 100

# Rate limiting and retry parameters
REQUEST_DELAY_SECONDS = 1.0
MAX_RETRIES = 5
BASE_BACKOFF_SECONDS = 2.0

all_comments = []

# Function to fetch JSON data with retry logic
def get_json(url, params, max_retries=MAX_RETRIES):
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=30)
            print("Status code:", response.status_code)

            if response.status_code == 200:
                return response.json()

            if response.status_code not in {429, 500, 502, 503, 504}:
                print("Non-retryable error:", response.text[:300])
                return {"data": []}

            delay = BASE_BACKOFF_SECONDS * (2 ** (attempt - 1))
            print(f"Temporary error. Retry {attempt}/{max_retries} in {delay:.1f} seconds.")
            time.sleep(delay)

        except Exception as e:
            delay = BASE_BACKOFF_SECONDS * (2 ** (attempt - 1))
            print(f"Request failed: {e}")
            print(f"Retry {attempt}/{max_retries} in {delay:.1f} seconds.")
            time.sleep(delay)

    return {"data": []}


# Function to fetch posts from a subreddit
def fetch_posts(subreddit):
    params = {
        "subreddit": subreddit,
        "size": posts_per_subreddit,
        "sort": "desc",
        "sort_type": "created_utc",
    }

    return get_json(SUBMISSION_URL, params).get("data", [])

# Function to fetch comments for a given post
def fetch_comments_for_post(post_id):
    params = {
        "link_id": post_id,
        "size": comments_per_post,
        "sort": "asc",
        "sort_type": "created_utc",
    }

    return get_json(COMMENT_URL, params).get("data", [])


for subreddit in subreddits:
    print(f"\nFetching posts from r/{subreddit}...")

    posts = fetch_posts(subreddit)
    print(f"Found {len(posts)} posts from r/{subreddit}")

    for post in posts:
        post_id = post.get("id")
        post_title = post.get("title")

        if not post_id:
            continue

        print(f"Collecting comments for post {post_id} from r/{subreddit}")

        comments = fetch_comments_for_post(post_id)
        print(f"Collected {len(comments)} comments for post {post_id}")

        for comment in comments:
            all_comments.append(
                {
                    "subreddit": subreddit,
                    "post_id": post_id,
                    "post_title": post_title,
                    "comment_id": comment.get("id"),
                    "author": comment.get("author"),
                    "body": comment.get("body"),
                    "parent_id": comment.get("parent_id"),
                    "link_id": comment.get("link_id"),
                    "score": comment.get("score"),
                    "created_utc": comment.get("created_utc")
                    if comment.get("created_utc") is not None
                    else comment.get("created"),
                }
            )

        time.sleep(REQUEST_DELAY_SECONDS)

# Convert the collected comments into a DataFrame and save to CSV
df = pd.DataFrame(all_comments)

df["created_datetime"] = pd.to_datetime(
    df["created_utc"],
    unit="s",
    errors="coerce",
)

df = df.sort_values(
    by=["subreddit", "post_id", "created_utc"],
    ascending=[True, True, True],
)

df.to_csv(
    "../data/reddit_technology_programming_comments.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nFinal dataset shape:", df.shape)
print("Dataset saved successfully to the data folder.")


Fetching posts from r/programming...
Status code: 200
Found 50 posts from r/programming
Status code: 200
Collected 2 comments for post 1kiea0u
Status code: 200
Collected 0 comments for post 1kie8k9
Status code: 200
Collected 0 comments for post 1kie3u9
Status code: 200
Collected 0 comments for post 1kidzyn
Status code: 200
Collected 0 comments for post 1kidq83
Status code: 200
Collected 0 comments for post 1kidcmt
Status code: 200
Collected 1 comments for post 1kid4sj
Status code: 200
Collected 0 comments for post 1kicz2q
Status code: 200
Collected 0 comments for post 1kic13i
Status code: 200
Collected 9 comments for post 1kibtfq
Status code: 200
Collected 0 comments for post 1kiba72
Status code: 200
Collected 2 comments for post 1ki97ud
Status code: 200
Collected 0 comments for post 1ki8d0f
Status code: 200
Collected 0 comments for post 1ki86dz
Status code: 200
Collected 0 comments for post 1ki6x1t
Status code: 200
Collected 1 comments for post 1ki6ftt
Status code: 200
Collected 0 co